In [ ]:
%pip install -r requirements.txt

In [17]:
import torch
import torch.nn as nn

# Q1 - creation of a simple transformer
model = nn.Transformer(d_model=512,
                        nhead=8,
                        num_encoder_layers=6,
                        num_decoder_layers=6)
print(model)

/home/shuptik/pw/sem6/ann/Transformers/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, o

In [26]:
import math
import torch
import torch.nn as nn

class TokenEmbedding(nn.Module):
    def __init__(self, embedding_dim: int, vocab_size: int) -> None:
        super().__init__()
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.embedding(x) * math.sqrt(self.embedding_dim) # this is to properly scale token embedding wrt to positional embedding

In [ ]:
import math
import torch.nn as nn
import torch

class PositionalEncoding(nn.Module):
    def __init__(self, block_size: int, embedding_dim: int) -> None:
        super().__init__()
        positional_embeddings = torch.zeros(block_size, embedding_dim)
        for pos in range(block_size):
            for index in range(embedding_dim // 2):
                denominator: float = 10000 ** (2 * index / embedding_dim)
                positional_embeddings[pos, 2 * index] = math.sin(pos / denominator)
                positional_embeddings[pos, 2 * index + 1] = math.cos(pos / denominator)
        self.register_buffer("positional_embeddings", positional_embeddings)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.positional_embeddings[:x.size(dim = 1), :]
    
vocab_size = 10000
embedding_dim = 512
sequence_length = 10
batch_size = 2
token_embedding = TokenEmbedding(embedding_dim=embedding_dim, vocab_size=vocab_size)
positional_embedding = PositionalEncoding(block_size=100, embedding_dim=embedding_dim)
id_tensor: torch.Tensor = torch.randint(low = 0, high=10, size = (batch_size, sequence_length))
result = positional_embedding(token_embedding(id_tensor))
print(result.shape)
print(result)

torch.Size([2, 10, 512])
tensor([[[-39.2093,  11.2686,  19.5650,  ...,   4.2863,  -0.4449, -13.6909],
         [ 17.8550, -29.1278, -12.4539,  ..., -22.0544,   7.4580,  -3.4072],
         [ 17.9229, -30.0843, -12.3393,  ..., -22.0544,   7.4581,  -3.4072],
         ...,
         [ 17.6705, -28.9142, -12.8234,  ..., -22.0544,   7.4586,  -3.4072],
         [ -4.1032, -24.7343,  24.0613,  ...,  13.3885, -22.4846,  29.4714],
         [  0.2026,  29.4657,   6.2753,  ...,  -4.0697, -33.1067,  11.7505]],

        [[ 24.8804, -17.8987, -28.1504,  ...,  23.1847,  49.8950,  -4.8838],
         [ 17.8550, -29.1278, -12.4539,  ..., -22.0544,   7.4580,  -3.4072],
         [ 38.1480, -11.4348, -26.7075,  ..., -19.1165,  12.9579,  19.6351],
         ...,
         [-35.8308, -15.4345, -13.5292,  ...,  20.3587, -11.6944,   4.6977],
         [-35.4984, -16.3339, -12.9910,  ...,  20.3587, -11.6943,   4.6977],
         [ -4.6804, -25.4999,  23.7470,  ...,  13.3885, -22.4845,  29.4714]]],
       grad_fn=<Add

In [29]:
import torch.nn.functional as F

class AttentionHead(nn.Module):
    def __init__(self, head_size: int) -> None:
        super().__init__()
        self.head_size = head_size
        self.query = nn.Linear(in_features=head_size, out_features=head_size, bias=False)
        self.key = nn.Linear(in_features = head_size, out_features=head_size, bias=False)
        self.values = nn.Linear(in_features = head_size, out_features=head_size, bias=False)
    def forward(self, x: torch.Tensor):
        (B, S, D) = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.values(x)
        weights =  F.softmax((q @ k.transpose(-2, -1))*self.head_size ** -0.5, dim = -1) # (B,S,S) shape
        scores = weights @ v # (B,S,D)
        return scores


class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads: int, embedding_dim: int) -> None:
        super().__init__()
        self.n_heads = n_heads
        self.embedding_dim = embedding_dim
        self.head_dim = embedding_dim // n_heads
        self.attention_heads = nn.ModuleList(
            AttentionHead(self.head_dim)
            for _ in range(n_heads)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        head_outputs = []
        for index in range(self.n_heads):
            start = index * self.head_dim
            end = start + self.head_dim
            x_head = x[:, :, start:end]
            attention = self.attention_heads[index](x_head)
            head_outputs.append(attention)
        return torch.cat(head_outputs, dim=-1)
    
n_heads = 8
embedding_dim = 512
attention_block = MultiHeadAttention(n_heads=n_heads, embedding_dim=embedding_dim)
input = result
result = attention_block(input)
print(result.shape)
print(result.shape == input.shape)


torch.Size([2, 10, 512])
True
